# Fine-tune Sign2Say และประเมินวิดีโอจริง

Notebook นี้เรียกโค้ดจริงจาก `DSI490-Sign2Say/GCP/script/job_entry.py`, `main.py`, `modules/pipeline.py` และ `modules/decode.py` โดยตรง ไม่มีข้อมูลสุ่ม ไม่มี mock metric และไม่แก้ `models/` production โดยอัตโนมัติ

ก่อนรันต้องมี: (1) prepared shards: `pose_store.npz`, `clips.parquet`; (2) base weights; (3) 5 วิดีโอจริงพร้อม gloss อ้างอิง; (4) เวลาเริ่ม-จบคำ หากต้องการ Segment F1/Recall แบบจับคู่เวลา. ห้ามใช้วิดีโอ evaluation นี้ใน train หรือจูน threshold.

In [ ]:
from pathlib import Path
import json, os, sys, time, subprocess, hashlib

REPO = Path('/Users/thawaphorn/Pee4/Term1/DSI490/DSI490-Sign2Say').resolve()
EXP = Path('/Users/thawaphorn/Pee4/Term1/DSI490/การทดลอง/finetune_real').resolve()
# เปลี่ยนสอง path นี้ให้เป็น prepared training data และ base weights จริง ก่อน run fine-tune
TRAIN_DATA = Path('CHANGE_ME/prepared_data')
BASE_MODELS = REPO / 'artifacts' / 'pretrained'
RUN_ID = time.strftime('enc-real-%Y%m%d-%H%M%S')
RUN_DIR = EXP / 'runs' / RUN_ID
DEVICE = 'auto'       # auto | mps | cuda | cpu
STEPS = 3600          # กำหนดจาก validation ไม่ใช่ผล mock
EVAL_EVERY = 400
LR_GCN, LR_MT5, LR_HEAD = 1e-4, 2e-5, 5e-4
assert REPO.is_dir() and (REPO/'GCP/script/job_entry.py').is_file()
EXP.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO))

## 1. ตรวจ source และข้อมูลก่อนฝึก

Fine-tune ใช้ `stage_train_encoder()` จาก `GCP/script/job_entry.py`: อ่าน `pose_store.npz` และ `clips.parquet`, ใช้เฉพาะ clips ที่ `role == train`, ประเมิน held-out signer ทุก `eval_every` steps. ด้านล่างตรวจ file contract ก่อน จึงไม่สร้างหรือแทนที่ข้อมูลเทียมให้เอง.

In [ ]:
required_data = ['pose_store.npz', 'clips.parquet']
required_models = ['unisign_wlasl_enc.pt', 'mt5_config.json']
missing = [str(TRAIN_DATA/x) for x in required_data if not (TRAIN_DATA/x).is_file()]
missing += [str(BASE_MODELS/x) for x in required_models if not (BASE_MODELS/x).is_file()]
if missing:
    raise FileNotFoundError('ยังไม่พร้อม fine-tune; ตั้ง TRAIN_DATA/BASE_MODELS ให้ถูกต้อง:\n' + '\n'.join(missing))
import pandas as pd
clips = pd.read_parquet(TRAIN_DATA/'clips.parquet')
assert {'clip_id', 'concept', 'signer', 'role', 'kind', 'view', 'span_f0', 'span_f1'} <= set(clips.columns)
display(clips.groupby(['role', 'kind']).size().rename('clips').reset_index())
assert (clips.role == 'train').any(), 'ไม่มี train split'
assert (clips.role != 'train').any(), 'ไม่มี held-out validation/test split'

## 2. Fine-tune encoder จริง

เซลล์นี้เรียก `.py` ต้นฉบับผ่าน subprocess และบันทึก checkpoint/history ลง `การทดลอง/finetune_real/runs/<RUN_ID>` เท่านั้น ไม่เขียนทับ `models/encoder.pt`. ตรวจ command ที่พิมพ์ออกมาก่อนรัน โดยเฉพาะจำนวน `STEPS` และ device.

In [ ]:
train_cmd = [sys.executable, str(REPO/'GCP/script/job_entry.py'),
    '--stage', 'train_encoder', '--data', str(TRAIN_DATA), '--models', str(BASE_MODELS),
    '--out', str(RUN_DIR), '--run_id', RUN_ID, '--device', DEVICE,
    '--steps', str(STEPS), '--eval_every', str(EVAL_EVERY),
    '--lr_gcn', str(LR_GCN), '--lr_mt5', str(LR_MT5), '--lr_head', str(LR_HEAD)]
print(' '.join(train_cmd))
# ลบ # บรรทัดถัดไปเมื่อยืนยัน command แล้ว; คำสั่งนี้ใช้ข้อมูลจริงและอาจใช้เวลานาน
# subprocess.run(train_cmd, cwd=REPO, check=True)

## 3. Train sequence/tagger และสร้าง artifacts ของ run

หลัง encoder train จบ ต้องใช้ encoder run เดียวกันสร้าง tagger/span head/bank ก่อนประเมิน มิฉะนั้น embedding space ไม่ตรงกัน. โค้ดนี้เรียก entry point ของ repo; output อยู่ใน experiment run ไม่แตะ deployable ONNX.

In [ ]:
assert RUN_DIR.is_dir(), 'รัน fine-tune สำเร็จก่อน'
SEQ_DIR = RUN_DIR.parent / f'seq-{RUN_ID}'
seq_cmd = [sys.executable, str(REPO/'main.py'), 'train-seq', '--encoder_run', str(RUN_DIR), '--out', str(SEQ_DIR), '--device', DEVICE]
print(' '.join(seq_cmd))
# subprocess.run(seq_cmd, cwd=REPO, check=True)

## 4. สร้าง model directory เฉพาะ run

ไม่แทนที่ `DSI490-Sign2Say/models/`: สร้างสำเนา minimal สำหรับ run นี้ แล้วใส่ `encoder_best.pt`, tagger, span head, null prototype และ bank ที่มีเฉพาะ training clips. การประเมินจึงไม่ใช้ embedding ของ held-out/evaluation clips เป็น prototype. ใช้ backend `torch` เพราะยังไม่ได้ export ONNX ของ run นี้.

In [ ]:
import shutil, numpy as np
EVAL_MODEL_DIR = RUN_DIR/'models_for_evaluation'
assert (RUN_DIR/'encoder_best.pt').is_file() and (RUN_DIR/'emb.npy').is_file()
assert (SEQ_DIR/'tagger.pt').is_file() and (SEQ_DIR/'span_head.npz').is_file() and (SEQ_DIR/'null_proto.npy').is_file()
if EVAL_MODEL_DIR.exists():
    raise FileExistsError(f'{EVAL_MODEL_DIR} already exists; use a new RUN_ID instead of overwriting evidence')
shutil.copytree(REPO/'models', EVAL_MODEL_DIR)
shutil.copy2(RUN_DIR/'encoder_best.pt', EVAL_MODEL_DIR/'encoder.pt')
for name in ('tagger.pt', 'span_head.npz', 'null_proto.npy'):
    shutil.copy2(SEQ_DIR/name, EVAL_MODEL_DIR/name)
index = pd.read_parquet(RUN_DIR/'emb_index.parquet'); Z = np.load(RUN_DIR/'emb.npy').astype(np.float32)
concepts = json.loads((RUN_DIR/'concept_index.json').read_text()); sign = index.concept.notna().to_numpy() & (index.role == 'train').to_numpy()
assert len(index) == len(Z) and sign.any()
np.savez(EVAL_MODEL_DIR/'bank.npz', Z=Z[sign].astype(np.float16), c=index.loc[sign, 'c'].astype(np.int32).to_numpy(),
         clip=index.loc[sign, 'clip_id'].astype(str).to_numpy(), dataset=index.loc[sign, 'dataset'].astype(str).to_numpy(), signer=index.loc[sign, 'signer'].astype(str).to_numpy())
(EVAL_MODEL_DIR/'concepts.json').write_text(json.dumps(concepts, ensure_ascii=False))
old_vocab = json.loads((REPO/'models/vocab_conversation.json').read_text()); present = set(index.loc[sign, 'concept'].dropna())
old_vocab['concepts'] = [w for w in old_vocab['concepts'] if w in present]
(EVAL_MODEL_DIR/'vocab_conversation.json').write_text(json.dumps(old_vocab, ensure_ascii=False, indent=2))
cfg = json.loads((EVAL_MODEL_DIR/'config.json').read_text()); cfg.update({'encoder_run':RUN_DIR.name, 'sequence_run':SEQ_DIR.name, 'evaluation_bank':'train_only'})
(EVAL_MODEL_DIR/'config.json').write_text(json.dumps(cfg, ensure_ascii=False, indent=2))
print(EVAL_MODEL_DIR, 'bank clips=', int(sign.sum()), 'conversation concepts=', len(old_vocab['concepts']))

## 5. ระบุวิดีโอจริง 5 คลิปและ ground truth

สร้าง/แก้ `five_video_manifest.json` ด้วย path วิดีโอจริง, `reference_words` ตามลำดับภาษามือจริง, และ `reference_segments` เป็น `{word,start_s,end_s}`. ถ้าไม่มีเวลา จะไม่คำนวณ Segment F1 และ IoU-matched Recall เพื่อไม่สร้างตัวเลขปลอม.

In [ ]:
MANIFEST = EXP/'five_video_manifest.json'
if not MANIFEST.exists():
    template = [{'id': f'video_{i:02d}', 'video': '', 'reference_words': [], 'reference_segments': [], 'evaluation_locked': True} for i in range(1, 6)]
    MANIFEST.write_text(json.dumps(template, ensure_ascii=False, indent=2))
    raise RuntimeError(f'สร้าง {MANIFEST}; กรอกข้อมูลจริงทั้ง 5 คลิป แล้วรันเซลล์นี้ใหม่')
items = json.loads(MANIFEST.read_text())
assert len(items) == 5 and len({x['id'] for x in items}) == 5
for x in items:
    assert x['evaluation_locked'] is True and Path(x['video']).is_file() and x['reference_words'], x['id']
    if x['reference_segments']:
        assert [s['word'] for s in x['reference_segments']] == x['reference_words']
pd.DataFrame(items)[['id','video','reference_words','reference_segments']]

## 6. Inference และ metrics บนวิดีโอจริง

ใช้ `SignPipeline` และ `modules.decode` เดิม: WER, Segment F1@IoU 0.3, Recall@1/5, accepted words correct, count error และ latency. ปิด LLM/TTS เพื่อวัด recognition; `candidate.score` เป็น retrieval score ไม่ใช่ probability ของคำ ส่วน `p_correct` เป็น calibrated confidence ของ segment.

In [ ]:
from modules.pipeline import SignPipeline
from modules.decode import wer, sentence_metrics, aggregate
EVAL_DIR = RUN_DIR/'five_video_evaluation'; EVAL_DIR.mkdir(parents=True, exist_ok=False)
assert EVAL_MODEL_DIR.is_dir(), 'สร้าง model directory ของ fine-tuned run ก่อน'
pipe = SignPipeline(model_dir=EVAL_MODEL_DIR, backend='torch', device=DEVICE, vocab='conversation', use_llm=False, use_face=True)
rows, raw_results, timed = [], {}, []
for item in items:
    t0 = time.perf_counter(); result = pipe.run(Path(item['video']), EVAL_DIR/item['id'], tts=False); latency = time.perf_counter()-t0
    raw_results[item['id']] = result
    pred = [s for s in result['segments'] if s['status'] != 'transition']
    row = {'id': item['id'], 'WER': float(wer(item['reference_words'], [s['nearest'] for s in pred])),
           'count_error': abs(len(pred)-len(item['reference_words'])), 'latency_s': latency,
           'accepted_total': sum(s['status']=='accepted' for s in pred), 'accepted_correct': None,
           'Segment_F1': None, 'Recall@1': None, 'Recall@5': None}
    if item['reference_segments']:
        gt = [(x['start_s']*result['fps'], x['end_s']*result['fps']) for x in item['reference_segments']]
        m = sentence_metrics(pred, gt, item['reference_words']); timed.append(m)
        row.update({'Segment_F1':m['seg_f1'], 'Recall@1':m['hit1']/m['n_inv'], 'Recall@5':m['hit5']/m['n_inv'],
                    'accepted_correct':sum(ok for ok, s in zip(m['correct'], pred) if s['status']=='accepted')})
    rows.append(row)
metrics = pd.DataFrame(rows); display(metrics)
summary = {'corpus_WER': sum(x['WER']*len(i['reference_words']) for x,i in zip(rows,items))/sum(len(i['reference_words']) for i in items),
           'mean_count_error': metrics['count_error'].mean(), 'mean_latency_s': metrics['latency_s'].mean(),
           'timed_videos': len(timed), 'repo_aggregate': aggregate(timed) if timed else None}
(EVAL_DIR/'metrics.json').write_text(json.dumps({'per_video':rows,'summary':summary}, ensure_ascii=False, indent=2)); summary

## 7. ตรวจหลักฐานรูปมือจริงต่อ segment (ยังไม่ rerank)

ใช้ `experiments/hybrid_pipeline/gesture_detector.py` ที่มีอยู่ ตรวจจาก MediaPipe landmarks ว่าแต่ละมือมีหลักฐาน `four_fingers_extended`, `four_fingers_curled`, `index_pointing`, และ `prayer_like_2d` หรือข้อมูลไม่พอ. ผลนี้เป็น diagnostic เท่านั้น: 2D ยังยืนยันกำหมัด/ทิศฝ่ามือ/การสัมผัสไม่ได้ จึงไม่ตัดคำออกหรือเปลี่ยน prediction จนกว่าจะมี annotation และผล validation ว่าช่วยจริง.

In [ ]:
import importlib.util
gesture_path = REPO/'experiments/hybrid_pipeline/gesture_detector.py'
spec = importlib.util.spec_from_file_location('gesture_detector', gesture_path)
gesture = importlib.util.module_from_spec(spec); spec.loader.exec_module(gesture)
evidence = []
for item in items:
    raw = pipe.landmarks_from_video(Path(item['video']))
    for seg in raw_results[item['id']]['segments']:
        g = gesture.detect_gestures(raw, seg['t0'], seg['t1'])
        evidence.append({'video':item['id'], 't0':seg['t0'], 't1':seg['t1'], 'prediction':seg['nearest'],
                         'p_correct':seg.get('p_correct'), 'hand_evidence':g['labels']['four_fingers_extended'],
                         'curled_evidence':g['labels']['four_fingers_curled'],
                         'pointing_evidence':g['labels']['index_pointing'],
                         'prayer_like_2d':g['labels']['prayer_like_2d']['status']})
(EVAL_DIR/'hand_evidence.json').write_text(json.dumps(evidence, ensure_ascii=False, indent=2))
pd.DataFrame(evidence)[['video','t0','t1','prediction','p_correct','prayer_like_2d']]

## 8. Visualize ต่อ segment

สร้าง HTML ต่อวิดีโอ: เฟรมกลางช่วง, เวลา, predicted word, `p_correct`, Top-5 retrieval score/support และหลักฐานมือ. ไฟล์อยู่ใน `EVAL_DIR`; ใช้ในการประชุมและตรวจคำผิดรายช่วง.

In [ ]:
import base64, html, cv2
by_video = {x['video']: [] for x in evidence}
for e in evidence: by_video[e['video']].append(e)
for item in items:
    cap = cv2.VideoCapture(item['video']); cards=[]
    try:
        for seg, ev in zip(raw_results[item['id']]['segments'], by_video[item['id']]):
            cap.set(cv2.CAP_PROP_POS_MSEC, 500*(seg['t0']+seg['t1'])); ok, frame = cap.read(); img=''
            if ok:
                ok, enc=cv2.imencode('.jpg',frame); img='<img width=240 src="data:image/jpeg;base64,'+base64.b64encode(enc).decode()+'">' if ok else ''
            top5=''.join(f"<li>{html.escape(c['word'])}: score={c.get('score')}, support={c.get('support')}</li>" for c in seg['candidates'][:5])
            cards.append(f"<article><b>{seg['t0']:.2f}–{seg['t1']:.2f}s · {html.escape(seg['nearest'])}</b><br>p_correct={seg.get('p_correct')} · {seg['status']}<br>{img}<ol>{top5}</ol><small>extended={ev['hand_evidence']}; curled={ev['curled_evidence']}; pointing={ev['pointing_evidence']}; prayer={ev['prayer_like_2d']}</small></article>")
    finally: cap.release()
    page='<meta charset="utf-8"><style>article{display:inline-block;vertical-align:top;width:280px;margin:8px;padding:8px;border:1px solid #bbb}</style><h1>'+html.escape(item['id'])+'</h1>'+''.join(cards)
    (EVAL_DIR/f"{item['id']}_segments.html").write_text(page)
print(EVAL_DIR)

## 9. เกณฑ์สรุปผล

รายงาน `metrics.json` พร้อม manifest, model artifacts และ HTML visualization ของ run เดียวกัน. เปรียบเทียบ baseline กับ fine-tuned run ได้เมื่อใช้ 5 คลิป, ground truth, vocabulary, backend และ metric script เดียวกัน. ห้ามสรุปว่า hybrid rules ช่วย จนกว่าจะวัด before/after บน validation ที่ไม่ใช้จูน rule และดูผลรายคำที่ผิด/ถูกพร้อมกัน.